In [ ]:
%cd ..
import os
from typing import Any

import torch
from tqdm.auto import tqdm
import librosa

import jiwer
from tqdm.autonotebook import tqdm
import librosa
import re
import string

from src.datasets_list import load_wikipedia_asr_splitted 
from src.asr_eval.utils.types import FLOATS
from src.asr_eval.models.voxtral_wrapper import VoxtralWrapper

print(f'{torch.cuda.is_available() = }')

In [ ]:
voxtral_asr = VoxtralWrapper(api="http://158.160.47.65:8000/v1", lang='ru', temperature=0.0, top_p=0.95, use_double_asr=True)
voxtral_no_asr = VoxtralWrapper(api="http://158.160.47.65:8000/v1", lang='ru', temperature=0.0, top_p=0.95, use_double_asr=False)


In [ ]:
d3 = load_wikipedia_asr_splitted()

In [ ]:


def preprocess_text(text: str) -> str:
    """Remove punctuation, whitespace and convert to lowercase"""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip().replace('\n', '')
    return text

voxtral_asr_no_prompt_wer = 0
voxtral_asr_with_prompt_wer = 0

voxtral_asr_no_prompt_cer = 0
voxtral_asr_with_prompt_cer = 0
 

a = []
b = []

n = 0

economics_words = [
    "экономика", "рынок", "деньги", "цена", "товар", "услуга", "спрос", "предложение", "конкуренция", "монополия",
    "олигополия", "конкурент", "потребитель", "производитель", "продавец", "покупатель", "торговля", "обмен", "стоимость", "ценность",
    "прибыль", "убыток", "доход", "расход", "выручка", "издержки", "себестоимость", "рентабельность", "эффективность", "производительность",
    "капитал", "инвестиции", "активы", "пассивы", "баланс", "бюджет", "дефицит", "профицит", "долг", "кредит",
    "банк", "банковский", "процент", "депозит", "вклад", "займ", "ссуда", "ипотека", "лизинг", "факторинг",
    "биржа", "акция", "облигация", "валюта", "курс", "девальвация", "ревальвация", "инфляция", "дефляция", "стагфляция",
    "ВВП", "ВНП", "национальный", "доход", "безработица", "занятость", "трудовые", "ресурсы", "заработная", "плата",
    "налог", "налогообложение", "пошлина", "тариф", "субсидия", "дотация", "льгота", "фискальная", "политика", "монетарная",
    "центральный", "банк", "денежная", "масса", "ликвидность", "платежеспособность", "кредитоспособность", "резервы", "золотовалютные", "резервы",
    "экспорт", "импорт", "торговый", "баланс", "платежный", "сальдо", "таможня", "квота", "эмбарго", "санкции",
    "глобализация", "интеграция", "ВТО", "ОПЕК", "МВФ", "всемирный", "международный", "валютный", "фонд", "группа",
    "макроэкономика", "микроэкономика", "эконометрика", "статистика", "индекс", "показатель", "коэффициент", "темп", "роста", "динамика",
    "циклы", "кризис", "рецессия", "депрессия", "оживление", "подъем", "бум", "спад", "восстановление", "стабилизация",
    "приватизация", "национализация", "либерализация", "дерегулирование", "реформы", "транзитная", "экономика", "постсоциалистическая", "трансформация", "переходная",
    "планирование", "государственное", "регулирование", "вмешательство", "невидимая", "рука", "рыночный", "механизм", "саморегулирование", "равновесие",
    "эластичность", "замещение", "дополняемость", "полезность", "предельная", "оптимизация", "максимизация", "минимизация", "ограничения", "бюджетное",
    "кривая", "спроса", "предложения", "безразличия", "производственных", "возможностей", "лаффера", "филлипса", "изокванта", "изокоста",
    "фирма", "предприятие", "корпорация", "акционерное", "общество", "товарищество", "кооператив", "индивидуальный", "предприниматель", "малый",
    "бизнес", "стартап", "венчурный", "капитал", "инновации", "технологии", "патент", "лицензия", "франшиза", "брендинг",
    "маркетинг", "реклама", "сегментация", "позиционирование", "конкурентное", "преимущество", "дифференциация", "себестоимость", "ценообразование", "демпинг",
    "страхование", "риск", "хеджирование", "деривативы", "фьючерсы", "опционы", "свопы", "форвардные", "контракты", "спекуляция",
    "аудит", "бухгалтерский", "учет", "отчетность", "финансовая", "амортизация", "износ", "основные", "средства", "оборотные",
    "логистика", "снабжение", "сбыт", "дистрибуция", "цепочка", "поставок", "склад", "транспорт", "аутсорсинг", "инсорсинг",
    "человеческий", "капитал", "образование", "квалификация", "мотивация", "стимулирование", "производительность", "труда", "разделение", "специализация",
    "адам", "смит", "карл", "маркс", "джон", "кейнс", "милтон", "фридман", "пол", "самуэльсон", "джозеф", "стиглиц",
    "классическая", "школа", "неоклассическая", "кейнсианство", "монетаризм", "институциональная", "экономика", "австрийская", "школа", "чикагская",
    "неолиберализм", "социальная", "рыночная", "командная", "смешанная", "экономика", "капитализм", "социализм", "коммунизм", "феодализм",
    "собственность", "частная", "общественная", "государственная", "интеллектуальная", "земельная", "рента", "арендная", "плата", "процентная",
    "ставка", "дисконтирование", "капитализация", "аннуитет", "сложный", "процент", "простой", "текущая", "стоимость", "будущая"
]

for i, sample in tqdm(enumerate(d3['economics'])):
    if i > 200:
        break
    
    try:
        v_asr_no_prompt = voxtral_asr.transcribe(sample['audio']['array'])[0].text        
        category_prompt = f"Тема этого аудио Экономика. Вот слова, которые могут быть в аудио: {economics_words}. Используй их для улучшения транскрипции."
        v_asr_with_prompt = voxtral_asr.transcribe(sample['audio']['array'], prompt=category_prompt)[0].text
        
        
        v_asr_no_prompt = preprocess_text(v_asr_no_prompt)
        v_asr_with_prompt = preprocess_text(v_asr_with_prompt)


        a.append(v_asr_no_prompt)
        b.append(v_asr_with_prompt)

        tr = preprocess_text(sample['transcription'])
        
        voxtral_asr_no_prompt_wer += jiwer.wer(tr, v_asr_no_prompt)
        voxtral_asr_with_prompt_wer += jiwer.wer(tr, v_asr_with_prompt)
        
        voxtral_asr_no_prompt_cer += jiwer.cer(tr, v_asr_no_prompt)
        voxtral_asr_with_prompt_cer += jiwer.cer(tr, v_asr_with_prompt)
        
        n += 1
        
    except Exception as e:
        print(e)
        continue

if n > 0:
    print(f"Results for {n} samples:")
    print(f"Voxtral ASR (no prompt) - WER: {voxtral_asr_no_prompt_wer/n:.4f}, CER: {voxtral_asr_no_prompt_cer/n:.4f}")
    print(f"Voxtral ASR (with prompt) - WER: {voxtral_asr_with_prompt_wer/n:.4f}, CER: {voxtral_asr_with_prompt_cer/n:.4f}")
